# Tutorial: Combined Topic Modeling

(last updated 10-07-2022)

In this tutorial, we are going to use our **Combined Topic Model** to get the topics out of a collections of articles.

## Topic Models

Topic models allow you to discover latent topics in your documents in a completely unsupervised way. Just use your documents and get topics out.

## Contextualized Topic Models

![](https://raw.githubusercontent.com/MilaNLProc/contextualized-topic-models/master/img/logo.png)

What are Contextualized Topic Models? **CTMs** are a family of topic models that combine the expressive power of BERT embeddings with the unsupervised capabilities of topic models to get topics out of documents.

## Python Package

You can find our package [here](https://github.com/MilaNLProc/contextualized-topic-models).

![https://github.com/MilaNLProc/contextualized-topic-models/actions](https://github.com/MilaNLProc/contextualized-topic-models/workflows/Python%20package/badge.svg) ![https://pypi.python.org/pypi/contextualized_topic_models](https://img.shields.io/pypi/v/contextualized_topic_models.svg) ![https://pepy.tech/badge/contextualized-topic-models](https://pepy.tech/badge/contextualized-topic-models)

# **Before you start...**

If you have additional questions about these topics, follow the links:

- you need to work with languages different than English: [click here!](https://contextualized-topic-models.readthedocs.io/en/latest/language.html#language-specific)
- you can't get good results with topic models: [click here!](https://contextualized-topic-models.readthedocs.io/en/latest/faq.html#i-am-getting-very-poor-results-what-can-i-do)
- you want to load your own embeddings: [click here!](https://contextualized-topic-models.readthedocs.io/en/latest/faq.html#can-i-load-my-own-embeddings)


# Enabling the GPU

First, you'll need to enable GPUs for the notebook:

- Navigate to Edit→Notebook Settings
- select GPU from the Hardware Accelerator drop-down

[Reference](https://colab.research.google.com/notebooks/gpu.ipynb)

# Installing Contextualized Topic Models

First, we install the contextualized topic model library

# Data

We are going to need some data. You should upload a file with one document per line. We assume you haven't run any preprocessing script.

However, if you want to first test the model without uploading your data, you can simply use the test file I'm putting here

In [4]:
text_file = "dbpedia_sample_abstract_20k_unprep.txt" # EDIT THIS WITH THE FILE YOU UPLOAD

# Importing what we need

In [5]:
from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.preprocessing import WhiteSpacePreprocessingStopwords
import nltk

## Preprocessing

Why do we use the **preprocessed text** here? We need text without punctuation to build the bag of word. Also, we might want only to have the most frequent words inside the BoW. Too many words might not help.

In [6]:
from nltk.corpus import stopwords as stop_words
import pandas as pd

nltk.download('stopwords')

# documents = [line.strip() for line in open(text_file, encoding="utf-8").readlines()[0:2000]]

documents = pd.read_csv("../Lyrics_extraction/scraped_lyrics_no_metadata.csv", encoding='utf-8')['lyrics'].tolist()

stopwords = list(stop_words.words("english"))

sp = WhiteSpacePreprocessingStopwords(documents, stopwords_list=stopwords)
preprocessed_documents, unpreprocessed_corpus, vocab, retained_indices = sp.preprocess()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
preprocessed_documents[:2]

['intro ooh ooh oh oh oh ow ow wow wow wow ah ow ow ow verse buy new car bitch real tear bitch real even talk bitch fucking shit god pull shit skrrt shit real wearing shit bitch yeah yuh came bear yuh yeah put chair yuh cross play fair yuh god got em chains real make head like walk spend light fifty fifty please right bad bitch cute face nice titties five hundred saint yeah bitch bitch sucker action nah raised wild whole wild wild wild bought go faster skrrt niggas tryna playin catch might pull ghost gas got chorus got bank yeah god bank yeah god bank yeah god bank yeah god bank yeah god bank god got ready gun yeah fast ready gun yeah god ready gun yeah god ready gun yeah god ready gun yeah god ready gun god verse yeah dog yeah huh nah real dog straight got house hills dog wanna see body nigga get killed dog wet wanna nigga get killed dog wet killed dog real dog lil dog dog want dog chasin dog yeah right bitch like dog wet plus shoot like dog like dog fast chorus got bank yeah god bank

We don't discard the non-preprocessed texts, because we are going to use them as input for obtaining the contextualized document representations.

Let's pass our files with preprocess and unpreprocessed data to our `TopicModelDataPreparation` object. This object takes care of creating the bag of words for you and of obtaining the contextualized BERT representations of documents. This operation allows us to create our training dataset.

Note: Here we use the contextualized model "paraphrase-distilroberta-base-v1".


In [8]:
tp = TopicModelDataPreparation("all-mpnet-base-v2")

training_dataset = tp.fit(text_for_contextual=unpreprocessed_corpus, text_for_bow=preprocessed_documents)

d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\contextualized_topic_models\utils\data_preparation.py:57: UserWarning: the longest document in your collection has 1836 words, the model instead truncates to 128 tokens.
  warnings.warn(f"the longest document in your collection has {max_local_length} words, the model instead "


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Let's check the first ten words of the vocabulary

In [9]:
tp.vocab[:10]

array(['across', 'act', 'actin', 'action', 'ad', 'adam', 'admit', 'adore',
       'advice', 'afraid'], dtype=object)

## Training our Combined TM

Finally, we can fit our new topic model. We will ask the model to find 50 topics in our collection.

In [ ]:
ctm = CombinedTM(bow_size=len(tp.vocab), contextual_size=768, n_components=10, num_epochs=10)
ctm.fit(training_dataset) # run the model

Epoch: [10/10]	 Seen Samples: [29440/30000]	Train Loss: 1206.6968808381455	Time: 0:00:37.191826: : 10it [06:13, 37.33s/it]
Sampling: [20/20]: : 20it [13:35, 40.77s/it]


# Topics

After training, now it is the time to look at our topics: we can use the

```
get_topic_lists
```

function to get the topics. It also accepts a parameter that allows you to select how many words you want to see for each topic.

If you look at the topics, you will see that they all make sense and are representative of a collection of documents that comes from Wikipedia (general knowledge). Notice that the topics are in English, because we trained the model on English documents.

In [11]:
ctm.get_topic_lists(10)

[['person',
  'boogie',
  'color',
  'sweetest',
  'hoping',
  'ran',
  'nature',
  'summer',
  'remember',
  'sun'],
 ['oh', 'chorus', 'pre', 'uh', 'na', 'ah', 'make', 'huh', 'mind', 'wanna'],
 ['shawty',
  'boogie',
  'shorty',
  'dough',
  'chilli',
  'usher',
  'moved',
  'easier',
  'mix',
  'blessed'],
 ['wanna',
  'girl',
  'yeah',
  'got',
  'baby',
  'like',
  'cause',
  'chorus',
  'love',
  'come'],
 ['get', 'like', 'wanna', 'cause', 'girl', 'got', 'let', 'yeah', 'uh', 'fuck'],
 ['shorty',
  'shawty',
  'bear',
  'person',
  'crew',
  'cute',
  'sat',
  'doctor',
  'breakin',
  'blessed'],
 ['thing',
  'everyone',
  'verse',
  'mind',
  'working',
  'power',
  'everybody',
  'easier',
  'brooklyn',
  'solo'],
 ['shorty',
  'dough',
  'cute',
  'gas',
  'finna',
  'shawty',
  'ballin',
  'mo',
  'crew',
  'chains'],
 ['de',
  'every',
  'breakin',
  'summer',
  'ha',
  'notice',
  'flowers',
  'blessed',
  'trip',
  'place'],
 ['la', 'ooh', 'chorus', 'ya', 'da', 'let', 'need'

# Let's Draw!

We can use PyLDAvis to plot our topic in a nice and friendly manner :)

In [12]:
lda_vis_data = ctm.get_ldavis_data_format(tp.vocab, training_dataset, n_samples=10)

Sampling: [10/10]: : 10it [57:40, 346.09s/it]


In [13]:
import pyLDAvis as vis

lda_vis_data = ctm.get_ldavis_data_format(tp.vocab, training_dataset, n_samples=10)

ctm_pd = vis.prepare(**lda_vis_data)
vis.display(ctm_pd)

Sampling: [10/10]: : 10it [05:55, 35.59s/it]


# Topic Predictions

Ok now we can take a document and see which topic has been assigned to it. Results will obviously change with respect to the documents you are using. For example, let's predict the topic of the first preprocessed document that is talking about a peninsula.

In [14]:
topics_predictions = ctm.get_thetas(training_dataset, n_samples=5) # get all the topic predictions

Sampling: [5/5]: : 5it [02:59, 35.87s/it]


In [15]:
preprocessed_documents[0] # see the text of our preprocessed document

'intro ooh ooh oh oh oh ow ow wow wow wow ah ow ow ow verse buy new car bitch real tear bitch real even talk bitch fucking shit god pull shit skrrt shit real wearing shit bitch yeah yuh came bear yuh yeah put chair yuh cross play fair yuh god got em chains real make head like walk spend light fifty fifty please right bad bitch cute face nice titties five hundred saint yeah bitch bitch sucker action nah raised wild whole wild wild wild bought go faster skrrt niggas tryna playin catch might pull ghost gas got chorus got bank yeah god bank yeah god bank yeah god bank yeah god bank yeah god bank god got ready gun yeah fast ready gun yeah god ready gun yeah god ready gun yeah god ready gun yeah god ready gun god verse yeah dog yeah huh nah real dog straight got house hills dog wanna see body nigga get killed dog wet wanna nigga get killed dog wet killed dog real dog lil dog dog want dog chasin dog yeah right bitch like dog wet plus shoot like dog like dog fast chorus got bank yeah god bank 

In [16]:
import numpy as np
topic_number = np.argmax(topics_predictions[0]) # get the topic id of the first document

In [17]:
topic_number

4

In [18]:
ctm.get_topic_lists(5)[15]

['love', 'baby', 'oh', 'girl', 'want']

In [19]:
ctm.get_topic_lists(5)[topic_number] #and the topic should be about natural location/places/related things

['get', 'like', 'wanna', 'cause', 'girl']

# Save Our Model for Later Use

In [20]:
ctm.save(models_dir="./saved_models/")

d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\contextualized_topic_models\models\ctm.py:473: Warning: This is an experimental feature that we has not been fully tested. Refer to the following issue:https://github.com/MilaNLProc/contextualized-topic-models/issues/38
  warnings.warn("This is an experimental feature that we has not been fully tested. Refer to the following issue:"


In [21]:
# let's remove the trained model
# del ctm

In [22]:
from contextualized_topic_models.models.ctm import CombinedTM

# Initialize the model with matching parameters
ctm = CombinedTM(
    bow_size = len(tp.vocab),
    contextual_size = 768,
)

# Load the saved model
ctm.load("../Experiments/contextualized_topic_model_nc_20_tpm_0.0_tpv_0.95_hs_prodLDA_ac_(100, 100)_do_softplus_lr_0.2_mo_0.002_rp_0.99/",
         epoch=9)

d:\Code\LabResearch\Topic-Modeling-Recommendation\.venv311\Lib\site-packages\contextualized_topic_models\models\ctm.py:498: Warning: This is an experimental feature that we has not been fully tested. Refer to the following issue:https://github.com/MilaNLProc/contextualized-topic-models/issues/38
  warnings.warn("This is an experimental feature that we has not been fully tested. Refer to the following issue:"


In [24]:
ctm.get_topic_lists(10)

[['oh',
  'baby',
  'chorus',
  'way',
  'come',
  'gonna',
  'girl',
  'love',
  'want',
  'right'],
 ['ballin',
  'confess',
  'car',
  'dee',
  'wonderin',
  'barely',
  'excuse',
  'player',
  'scott',
  'lyrics'],
 ['na', 'chorus', 'ah', 'hey', 'want', 'girl', 'baby', 'let', 'get', 'ooh'],
 ['like',
  'ya',
  'chorus',
  'take',
  'good',
  'cause',
  'back',
  'make',
  'let',
  'work'],
 ['girl', 'got', 'like', 'yeah', 'know', 'oh', 'get', 'want', 'uh', 'baby'],
 ['baby',
  'love',
  'want',
  'gonna',
  'chorus',
  'girl',
  'ooh',
  'ya',
  'hey',
  'need'],
 ['bitch',
  'nigga',
  'fuck',
  'got',
  'like',
  'shit',
  'ass',
  'back',
  'money',
  'niggas'],
 ['fuck',
  'bitch',
  'let',
  'get',
  'nigga',
  'em',
  'shit',
  'ass',
  'niggas',
  'money'],
 ['la',
  'anyone',
  'freaky',
  'bom',
  'song',
  'children',
  'lie',
  'derulo',
  'bloody',
  'dare'],
 ['wonderin',
  'excuse',
  'redfoo',
  'cute',
  'xscape',
  'ballin',
  'warning',
  'ex',
  'coast',
  'sat']

In [25]:
print(len(tp.vocab))

1991
